# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* Unit of analysis:** One row = one unique content item per client (`client_id` x `content_id`) evaluated over a monthly window.
* Time Window:** March 2026 (`month = '2026-03'`) chosen as the mid-panel working window (leaving June 2026 sealed as the final test set).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Feature:** `gsc_avg_position`, `ctr_prev30`, `engagement_rate_prev30` — metrics known reliably before the prediction moment and properly windowed.
* **Label / Proxy:** `is_declining_label` — the target variable derived from future trend indicators. Never use as a feature.
* **Context:** `client_id`, `content_id` — used strictly for grouping, joining, and splits. Never learned from by the model.
* **Excluded:** Raw future performance metrics from the target window — excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

hf_token = userdata.get('HF_TOKEN').strip()

df = load_dataset("FlyRank/internship-warehouse", data_files="fact_content_daily_performance_sample.parquet", token=hf_token, split="train")
pdf = df.to_pandas()

pdf['report_date'] = pd.to_datetime(pdf['report_date'])
print(f"Sample Date range: {pdf['report_date'].min()} to {pdf['report_date'].max()}")

slice_df = pdf[(pdf['report_date'].dt.year == 2026) & (pdf['report_date'].dt.month == 6)]
print(f"Total rows in slice (2026-06): {len(slice_df)}")

if 'client_hash_id' in slice_df.columns and 'content_hash_id' in slice_df.columns:
    duplicates = slice_df.groupby(['client_hash_id', 'content_hash_id', 'report_date']).size().reset_index(name='count')
    print(f"Grain Violations (should be 0): {len(duplicates[duplicates['count'] > 1])}")

Sample Date range: 2026-06-01 00:00:00 to 2026-06-30 00:00:00
Total rows in slice (2026-06): 11694072
Grain Violations (should be 0): 6390


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Data Limits & Limitations:**
  * History depth varies per client, meaning earlier history is absent rather than zero.
  * Window overlaps between past features and target labels require strict alignment before joining.
  * The dataset cannot tell us unrecorded external traffic shifts outside search console metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.